# Experimentos — ronda 5 · mejores configuraciones + lluvia (CHIRPS) · AICOLOMBIA

Las rondas 1–4 dejaron como mejores a `E10S` (LightGBM simple + contexto espacial), `E10S_arboles_50`, el ensamble
`ENS_E10S_E7` y, como alternativa de algoritmo, CatBoost (`CB_d4`). Esta ronda les agrega **lluvia observada** de CHIRPS
y mide si aporta, con la misma lógica de siempre: cortes F1–F3, puntaje de validación, regla de mejorar **≥ 0.5 puntos
a E10S sin empeorar en F3** y registro central.

⚠ **Antes de entregar una variante con lluvia hay que tener la confirmación de los organizadores.** La regla prohíbe datos
anteriores o posteriores a las fechas del reto. Aquí solo se usan 2018-01 → 2025-12, los mismos meses del reto.

### Lluvia y transformaciones
Fuente: CHIRPS v2.0 mensual (Climate Hazards Center, UC Santa Barbara; Funk et al., 2015). Es el promedio de los
pixeles de ~5 km dentro de cada municipio (cartografía UPRA). Se lee de `lluvia_chirps_mensual.csv` (generado por
`scripts/externos/03_lluvia_chirps.py`); si no está, el notebook la descarga de CHIRPS por su cuenta.

| Variable | Qué mide | Por qué |
|---|---|---|
| `lluvia_log`, `lluvia_3m_log` | log(1 + mm) del mes y de los últimos 3 meses (mes actual + 2 anteriores) | Nivel de lluvia. El logaritmo reduce el peso de los valores extremos del Pacífico |
| `lluvia_anom` | log del mes − promedio del log del **mismo municipio y mismo mes del año** | ¿Llovió más o menos de lo normal **para ese lugar y esa época**? Quita el clima de fondo (que ya capturan el historial y el mes) y casi todo el sesgo de CHIRPS en montaña |
| `lluvia_3m_anom` | Lo mismo con el acumulado de 3 meses | Suelo saturado: los deslizamientos y crecientes responden a lluvias acumuladas |
| `lluvia_anom_rezago1` | Anomalía del mes anterior | Lluvia antecedente |
| `lluvia_anom_x_tasa` | Anomalía × tasa histórica del municipio en ese mes | La misma lluvia pesa más donde suele haber eventos. Ayuda a un modelo de solo 100 árboles |

El promedio "normal" (climatología) se calcula **solo con 2018-01 → 2022-09** (periodo de entrenamiento). La lluvia es
una covariable, no el objetivo, así que no hay fuga del objetivo. Los 2 primeros meses de 2018 no tienen acumulado de 3
meses (no se usan datos de 2017): quedan vacíos y LightGBM los maneja.

### Variantes
| Variante | Qué es |
|---|---|
| `E10S`, `E10S_arboles_50`, `E7_lento` | Referencias sin lluvia (deben reproducir las rondas 3–4) |
| `E10S_LL` | E10S + todas las variables de lluvia (nivel y anomalías) en los 4 modelos |
| `E10S_LL_anom` | E10S + solo las anomalías (sin el nivel) |
| `E10S_LL_50`, `E10S_LL_150` | E10S_LL con 50 o 150 árboles en el clasificador (¿la lluvia pide más o menos complejidad?) |
| `E10S_LL_placebo` | Igual que `E10S_LL_anom`, pero con la lluvia **del mismo mes del año anterior**. Si mejora tanto como la real, la ganancia no viene de la lluvia del mes |
| `E7_LL` | E7_lento (sin variables espaciales, con early stopping) + lluvia |
| `E10S_LL_solo_clasif` | Probabilidad de `E10S_LL` + magnitud y cuantiles de `E10S` (sin entrenar nada nuevo) |
| `E10S_LL_solo_magnitud` | Probabilidad de `E10S` + magnitud y cuantiles de `E10S_LL` |
| `CB_d4_LL` | CatBoost de profundidad 4 como clasificador con lluvia; magnitud y cuantiles de `E10S_LL` |
| `ENS_E10S_E7` | Referencia: promedio fijo de E10S y E7_lento |
| `ENS_E10S_E7_LL` | Promedio fijo de `E10S_LL` y `E7_LL` |
| `ENS_E10S_E7_CB_LL` | Promedio fijo de `E10S_LL`, `E7_LL` y `CB_d4_LL` |

Los ensambles y las combinaciones se definieron **antes** de ver los resultados.

**Cómo correrlo en el PC:** abrir este notebook desde la carpeta `notebooks/` del repo (allí están `datos/` y la caché
de la cartografía) y ejecutar todo. Al final se actualiza `outputs/reportes/registro_experimentos.csv` y quedan los CSV
en `entregas_ronda5/`.

## 1. Datos, tratamiento, historial, cartografía y vecindad (igual que las rondas 3–4)

In [1]:
import os, zipfile
import numpy as np
import pandas as pd

URL_DATOS = "PEGUE_AQUI_EL_ENLACE_DE_LOS_DATOS"  # enlace de Google Drive al archivo hackathon_datos.zip (se comparte el día del evento)
DATA_DIR = os.environ.get("HACKATHON_DATA", "datos")

if not os.path.exists(os.path.join(DATA_DIR, "entrenamiento.csv")):
    if URL_DATOS:
        import gdown
        import re
        ident = re.search(r"/d/([\w-]+)", URL_DATOS)   # enlace directo: funciona con cualquier versión de gdown
        url = f"https://drive.google.com/uc?id={ident.group(1)}" if ident else URL_DATOS
        gdown.download(url, "hackathon_datos.zip", quiet=False)
    with zipfile.ZipFile("hackathon_datos.zip") as z:
        z.extractall(DATA_DIR)

leer = lambda f: pd.read_csv(os.path.join(DATA_DIR, f), dtype={"DIVIPOLA": str}, parse_dates=["fecha"])
train = leer("entrenamiento.csv")
test_pub = leer("prueba_equipos.csv")     # periodo de la tabla de posiciones en vivo
test_priv = leer("prueba_oculta.csv")     # periodo de la nota oficial
dic = pd.read_csv(os.path.join(DATA_DIR, "diccionario_datos.csv"))
print(train.shape, test_pub.shape, test_priv.shape)

(62814, 25) (20938, 13) (22040, 13)


In [2]:
ESPERADAS = {"entrenamiento": 62_814, "prueba_equipos": 20_938, "prueba_oculta": 22_040}
for nombre, df in {"entrenamiento": train, "prueba_equipos": test_pub, "prueba_oculta": test_priv}.items():
    print(f"{nombre:15s} filas={len(df):,} (esperadas {ESPERADAS[nombre]:,}) · municipios={df.DIVIPOLA.nunique()} · "
          f"llaves repetidas={df.duplicated(['DIVIPOLA', 'fecha']).sum()} · celdas nulas={int(df.isna().sum().sum())}")
nulos = pd.concat({"entrenamiento": train.isna().sum(), "prueba_equipos": test_pub.isna().sum(),
                   "prueba_oculta": test_priv.isna().sum()}, axis=1)
display(nulos[nulos.sum(axis=1) > 0])
fuera = pd.concat([test_pub, test_priv]).ONI.gt(train.ONI.max()).mean()
print(f"ONI de prueba por encima del máximo del entrenamiento: {fuera:.1%} de las filas (los árboles no extrapolan)")

entrenamiento   filas=62,814 (esperadas 62,814) · municipios=1102 · llaves repetidas=0 · celdas nulas=162
prueba_equipos  filas=20,938 (esperadas 20,938) · municipios=1102 · llaves repetidas=0 · celdas nulas=38
prueba_oculta   filas=22,040 (esperadas 22,040) · municipios=1102 · llaves repetidas=0 · celdas nulas=40


,entrenamiento,prueba_equipos,prueba_oculta
densidad_poblacional,12,0.0,0.0
ingresos_tributarios,36,0.0,0.0
inversion_gestion_riesgo,57,19.0,20.0
nbi_pct,57,19.0,20.0


ONI de prueba por encima del máximo del entrenamiento: 20.5% de las filas (los árboles no extrapolan)


In [3]:
COV_ESTATICAS = ["nbi_pct", "elevacion_media", "pendiente_media", "pct_pendiente_alta"]
COV_ANUALES = ["densidad_poblacional", "ingresos_tributarios", "inversion_gestion_riesgo"]
RANGOS = {"densidad_poblacional": (0, None), "ingresos_tributarios": (0, None), "inversion_gestion_riesgo": (0, None),
          "nbi_pct": (0, 100), "elevacion_media": (-50, 6000), "pendiente_media": (0, 90),
          "pct_pendiente_alta": (0, 100), "ONI": (-3.5, 3.5)}
HISTORIAL = ["n_eventos", "familias_desplazadas", "viv_destruidas", "viv_averiadas", "n_movimiento_masa",
             "n_inundacion", "n_vendaval", "n_creciente_subita", "n_avenida_torrencial", "n_otros"]
OBJETIVO = ["tiene_evento", "personas_desplazadas"]


def deduplicar(df):
    df = df.drop_duplicates()
    df = (df.assign(_nulos=df.isna().sum(axis=1)).sort_values(["DIVIPOLA", "fecha", "_nulos"])
            .drop_duplicates(["DIVIPOLA", "fecha"]).drop(columns="_nulos"))
    return df.sort_values(["DIVIPOLA", "fecha"]).reset_index(drop=True)


def preparar(train, test_pub, test_priv):
    partes = {}
    for k, df in {"train": train, "pub": test_pub, "priv": test_priv}.items():
        df = df.copy()
        df["DIVIPOLA"] = df["DIVIPOLA"].astype(str).str.strip().str.zfill(5)
        df["anio"], df["mes"] = df["fecha"].dt.year, df["fecha"].dt.month
        n0 = len(df)
        partes[k] = deduplicar(df)
        if len(partes[k]) != n0:
            print(f"{k}: {n0 - len(partes[k])} filas duplicadas eliminadas")

    tr = partes["train"]
    tr["personas_desplazadas"] = tr["personas_desplazadas"].clip(lower=0)
    tr[HISTORIAL] = tr[HISTORIAL].clip(lower=0).fillna(0)
    viv = tr[["viv_destruidas", "viv_averiadas"]].sum(axis=1)
    afectacion = (tr["personas_desplazadas"].fillna(0) > 0) | (viv > 0)
    corregidas = int(((tr["tiene_evento"] != 1) & afectacion).sum())
    tr["tiene_evento"] = np.where(afectacion, 1, tr["tiene_evento"].fillna(0)).astype(int)
    tr["personas_desplazadas"] = tr["personas_desplazadas"].fillna(0)
    print(f"Objetivo corregido según la definición oficial: {corregidas} filas")

    panel = pd.concat([d.assign(_parte=k, _i=np.arange(len(d))) for k, d in partes.items()], ignore_index=True)
    for c, (lo, hi) in RANGOS.items():
        malo = pd.Series(False, index=panel.index)
        if lo is not None:
            malo |= panel[c] < lo
        if hi is not None:
            malo |= panel[c] > hi
        if malo.any():
            print(f"{c}: {int(malo.sum())} valores imposibles -> nulo")
        panel.loc[malo, c] = np.nan
    faltaba = {c: panel[c].isna() for c in RANGOS if panel[c].isna().any()}

    panel["ONI"] = panel["ONI"].fillna(panel.groupby("fecha")["ONI"].transform("median"))
    for c in COV_ESTATICAS:
        panel[c] = panel[c].fillna(panel.groupby("DIVIPOLA")[c].transform("median"))
    for c in COV_ANUALES:   # el panel está en orden cronológico dentro de cada municipio
        panel[c] = panel[c].fillna(panel.groupby(["DIVIPOLA", "anio"])[c].transform("median"))
        panel[c] = panel.groupby("DIVIPOLA")[c].transform(lambda s: s.ffill().bfill())
    dep = panel["DIVIPOLA"].str[:2]
    for c in COV_ESTATICAS + COV_ANUALES:
        grupos = [dep, panel["anio"]] if c in COV_ANUALES else [dep]
        panel[c] = panel[c].fillna(panel.groupby(grupos)[c].transform("median")).fillna(panel[c].median())
    for c, mascara in faltaba.items():
        panel[c + "_imputado"] = mascara.astype(int)
        print(f"{c}: {int(mascara.sum()):,} valores imputados")
    panel["fase_enso"] = np.select([panel.ONI >= 0.5, panel.ONI <= -0.5], ["El Nino", "La Nina"], "Neutral")

    salida = {k: panel[panel._parte == k].sort_values("_i").drop(columns=["_parte", "_i"]).reset_index(drop=True)
              for k in partes}
    salida["train"]["tiene_evento"] = salida["train"]["tiene_evento"].astype(int)
    for k in ("pub", "priv"):
        salida[k] = salida[k].drop(columns=OBJETIVO + HISTORIAL, errors="ignore")
    return salida["train"], salida["pub"], salida["priv"]


train, test_pub, test_priv = preparar(train, test_pub, test_priv)
print(train.shape, test_pub.shape, test_priv.shape)

Objetivo corregido según la definición oficial: 0 filas
densidad_poblacional: 12 valores imputados
ingresos_tributarios: 36 valores imputados
inversion_gestion_riesgo: 96 valores imputados
nbi_pct: 96 valores imputados
(62814, 29) (20938, 17) (22040, 17)


In [4]:
K = {"dep": 120, "nm": 1102, "dm": 60, "mun": 24, "mm": 3, "mag": 3}
HIST_COLS = ["hist_tasa_mun", "hist_tasa_mes", "hist_tasa_dep", "hist_tasa_dep_mes", "hist_tasa_nac_mes",
             "hist_mag_mun", "hist_esperado_log", "hist_pct_inund", "hist_pct_masa", "hist_reciente_12"]


def _conteos(hist, claves, p):
    return (hist.groupby(claves).agg(**{f"e_{p}": ("tiene_evento", "sum"), f"n_{p}": ("tiene_evento", "size"),
                                         f"lp_{p}": ("logp", "sum")}).reset_index())


def historial_congelado(hist, filas):
    """Historial calculado con todo `hist` (que es pasado respecto a `filas`), aplicado a `filas`."""
    hist = hist.assign(logp=np.log1p(hist["personas_desplazadas"]), dep=hist["DIVIPOLA"].str[:2])
    f = filas.assign(dep=filas["DIVIPOLA"].str[:2])
    for claves, p in [(["DIVIPOLA"], "mun"), (["DIVIPOLA", "mes"], "mm"), (["dep"], "dep"),
                      (["dep", "mes"], "dm"), (["mes"], "nm")]:
        f = f.merge(_conteos(hist, claves, p), on=claves, how="left")
    c = f.fillna({col: 0 for col in f.columns if col[:2] in ("e_", "n_") or col.startswith("lp_")})
    tasa_nac = hist["tiene_evento"].mean()
    tasa_nm = (c.e_nm + K["nm"] * tasa_nac) / (c.n_nm + K["nm"])
    tasa_dep = (c.e_dep + K["dep"] * tasa_nac) / (c.n_dep + K["dep"])
    tasa_dm = (c.e_dm + K["dm"] * tasa_dep * tasa_nm / tasa_nac) / (c.n_dm + K["dm"])
    tasa_mun = (c.e_mun + K["mun"] * tasa_dep) / (c.n_mun + K["mun"])
    tasa_mm = (c.e_mm + K["mm"] * tasa_mun * tasa_dm / tasa_dep) / (c.n_mm + K["mm"])
    mag_nac = hist["logp"].sum() / max(hist["tiene_evento"].sum(), 1)
    mag_dep = (c.lp_dep + 5 * mag_nac) / (c.e_dep + 5)
    mag_mun = (c.lp_mun + K["mag"] * mag_dep) / (c.e_mun + K["mag"])

    tipos = hist.groupby("DIVIPOLA")[["n_inundacion", "n_movimiento_masa", "n_eventos"]].sum()
    reciente = hist[hist["fecha"] > hist["fecha"].max() - pd.DateOffset(months=12)].groupby("DIVIPOLA")["tiene_evento"].mean()
    salida = filas.copy()
    salida["hist_tasa_mun"] = tasa_mun.to_numpy()
    salida["hist_tasa_mes"] = tasa_mm.to_numpy()
    salida["hist_tasa_dep"] = tasa_dep.to_numpy()
    salida["hist_tasa_dep_mes"] = tasa_dm.to_numpy()
    salida["hist_tasa_nac_mes"] = tasa_nm.to_numpy()
    salida["hist_mag_mun"] = mag_mun.to_numpy()
    salida["hist_esperado_log"] = (tasa_mm * mag_mun).to_numpy()   # E[log1p(personas)] según la historia
    salida["hist_pct_inund"] = filas["DIVIPOLA"].map((tipos.n_inundacion + 1) / (tipos.n_eventos + 3)).fillna(1 / 3).to_numpy()
    salida["hist_pct_masa"] = filas["DIVIPOLA"].map((tipos.n_movimiento_masa + 1) / (tipos.n_eventos + 3)).fillna(1 / 3).to_numpy()
    salida["hist_reciente_12"] = filas["DIVIPOLA"].map(reciente).fillna(tasa_nac).to_numpy()
    return salida


def historial_entrenamiento(hist):
    """Filas de entrenamiento: para cada año Y (desde el segundo), historial con los años anteriores a Y."""
    anios = sorted(hist["anio"].unique())
    return pd.concat([historial_congelado(hist[hist.anio < a], hist[hist.anio == a]) for a in anios[1:]],
                     ignore_index=True)

In [5]:
import json
import math
import urllib.parse
import urllib.request

CARTO_SERVICIO = ("https://geoservicios.upra.gov.co/arcgis/rest/services/referencia_geografica/"
                  "municipios_generalizada/MapServer/0/query")
CARTO_GEOJSON = "cartografia_municipios.geojson"   # caché local, se crea en la primera corrida
CARTO_CSV = "geografia.csv"                        # alternativa sin internet
KM_POR_GRADO_LAT, KM_POR_GRADO_LON = 110.574, 111.320


def descargar_cartografia(tolerancia=0.002):
    parametros = {"where": "1=1", "outFields": "cod_dane_mpio,municipio,departamen", "outSR": 4326,
                  "maxAllowableOffset": tolerancia, "geometryPrecision": 4, "resultRecordCount": 2000, "f": "geojson"}
    with urllib.request.urlopen(f"{CARTO_SERVICIO}?{urllib.parse.urlencode(parametros)}", timeout=120) as r:
        return json.loads(r.read().decode("utf-8"))


def centroide_y_area(geometria):
    """Centroide del polígono exterior más grande y área en km² (proyección local equirectangular)."""
    poligonos = [geometria["coordinates"]] if geometria["type"] == "Polygon" else geometria["coordinates"]
    area_km2, mejor, mejor_area = 0.0, None, -1.0
    for poligono in poligonos:
        lat_media = sum(y for _, y in poligono[0]) / len(poligono[0])
        escala = KM_POR_GRADO_LAT * KM_POR_GRADO_LON * math.cos(math.radians(lat_media))
        for k, anillo in enumerate(poligono):
            a = cx = cy = 0.0
            for (x0, y0), (x1, y1) in zip(anillo, anillo[1:] + anillo[:1]):
                cruz = x0 * y1 - x1 * y0
                a, cx, cy = a + cruz, cx + (x0 + x1) * cruz, cy + (y0 + y1) * cruz
            a /= 2
            area_km2 += (abs(a) if k == 0 else -abs(a)) * escala
            if k == 0 and a != 0 and abs(a) > mejor_area:
                mejor_area, mejor = abs(a), (cx / (6 * a), cy / (6 * a))
    return mejor, area_km2


def tabla_desde_geojson(geojson):
    filas = []
    for f in geojson["features"]:
        if f.get("geometry"):
            (lon, lat), area = centroide_y_area(f["geometry"])
            filas.append({"DIVIPOLA": str(f["properties"]["cod_dane_mpio"]).zfill(5),
                          "lat": round(lat, 4), "lon": round(lon, 4), "area_km2": round(area, 1)})
    return pd.DataFrame(filas).drop_duplicates("DIVIPOLA").reset_index(drop=True)


def cargar_cartografia():
    geojson, origen = None, None
    if os.path.exists(CARTO_GEOJSON):
        with open(CARTO_GEOJSON, encoding="utf-8") as f:
            geojson, origen = json.load(f), f"caché local ({CARTO_GEOJSON})"
    else:
        try:
            geojson, origen = descargar_cartografia(), "servicio público de la UPRA (MGN del DANE)"
            with open(CARTO_GEOJSON, "w", encoding="utf-8") as f:
                json.dump(geojson, f)
        except Exception as error:
            print(f"⚠ No se pudo descargar la cartografía: {error}")
    if geojson is not None:
        return tabla_desde_geojson(geojson), geojson, origen
    if os.path.exists(CARTO_CSV):
        tabla = pd.read_csv(CARTO_CSV, dtype={"DIVIPOLA": str})[["DIVIPOLA", "lat", "lon", "area_km2"]]
        return tabla, None, f"archivo subido ({CARTO_CSV}; sin polígonos: no hay vecindad por frontera)"
    return None, None, "no disponible"


def validar_cartografia(tabla, datos):
    t = tabla.set_index("DIVIPOLA")
    codigos = set(datos["DIVIPOLA"])
    faltan = sorted(codigos - set(tabla["DIVIPOLA"]))
    dens = datos.loc[(datos["DIVIPOLA"] == "11001") & (datos["anio"] == datos["anio"].max()), "densidad_poblacional"]
    pob_bogota = float(dens.iloc[0] * t["area_km2"].get("11001", np.nan)) if len(dens) else float("nan")
    bloqueantes = [
        ("al menos 1,100 municipios", len(tabla) >= 1100),
        ("códigos DIVIPOLA únicos y de 5 dígitos",
         tabla["DIVIPOLA"].is_unique and tabla["DIVIPOLA"].str.fullmatch(r"\d{5}").all()),
        ("centroides dentro de Colombia",
         tabla["lat"].between(-4.5, 13.6).all() and tabla["lon"].between(-82.0, -66.8).all()),
        ("áreas positivas", (tabla["area_km2"] > 0).all()),
        ("área de Bogotá ≈ 1,636 km² (±5 %)", abs(t["area_km2"].get("11001", 0) / 1636 - 1) < 0.05),
    ]
    informativas = [
        (f"población de Bogotá = densidad × área ≈ 7–8 millones (da {pob_bogota / 1e6:.1f} M)", 5e6 < pob_bogota < 1e7),
        (f"cubre los {len(codigos):,} municipios del reto (faltan {len(faltan)}: {faltan[:5]})", not faltan),
    ]
    for texto, ok in bloqueantes + informativas:
        print(("✅ " if ok else "❌ ") + texto)
    return all(ok for _, ok in bloqueantes)


GEO_TABLA, GEO_JSON, origen = cargar_cartografia()
USAR_ESPACIAL = GEO_TABLA is not None and validar_cartografia(GEO_TABLA, train)
print(f"Cartografía: {origen} → variantes espaciales {'ACTIVADAS' if USAR_ESPACIAL else 'DESACTIVADAS'}")

✅ al menos 1,100 municipios
✅ códigos DIVIPOLA únicos y de 5 dígitos
✅ centroides dentro de Colombia
✅ áreas positivas
✅ área de Bogotá ≈ 1,636 km² (±5 %)
✅ población de Bogotá = densidad × área ≈ 7–8 millones (da 7.8 M)
✅ cubre los 1,102 municipios del reto (faltan 0: [])
Cartografía: caché local (cartografia_municipios.geojson) → variantes espaciales ACTIVADAS


In [6]:
def pares_knn(tabla, k, ponderado=False):
    lat, lon = np.radians(tabla["lat"].to_numpy()), np.radians(tabla["lon"].to_numpy())
    a = (np.sin((lat[:, None] - lat[None, :]) / 2) ** 2
         + np.cos(lat)[:, None] * np.cos(lat)[None, :] * np.sin((lon[:, None] - lon[None, :]) / 2) ** 2)
    distancia = 2 * np.arcsin(np.sqrt(np.clip(a, 0, 1)))
    np.fill_diagonal(distancia, np.inf)
    cercanos = np.argsort(distancia, axis=1)[:, :k]
    km = np.take_along_axis(distancia, cercanos, axis=1) * 6371
    peso = 1 / (km + 1) if ponderado else np.ones_like(km)
    codigos = tabla["DIVIPOLA"].to_numpy()
    return pd.DataFrame({"DIVIPOLA": np.repeat(codigos, k), "vecino": codigos[cercanos.ravel()], "peso": peso.ravel()})


def pares_frontera(geojson, respaldo):
    try:
        from shapely.geometry import shape
        from shapely.strtree import STRtree
    except ImportError:
        print("⚠ shapely no está disponible: se omite la vecindad por frontera")
        return None
    feats = [f for f in geojson["features"] if f.get("geometry")]
    codigos = np.array([str(f["properties"]["cod_dane_mpio"]).zfill(5) for f in feats])
    formas = [shape(f["geometry"]).buffer(0.002) for f in feats]      # tolera la simplificación de los límites
    i, j = STRtree(formas).query(formas, predicate="intersects")
    distintos = i != j
    pares = pd.DataFrame({"DIVIPOLA": codigos[i[distintos]], "vecino": codigos[j[distintos]], "peso": 1.0}).drop_duplicates()
    islas = sorted(set(codigos) - set(pares["DIVIPOLA"]))
    return pd.concat([pares, respaldo[respaldo["DIVIPOLA"].isin(islas)]], ignore_index=True)


VECINDADES = {}
if USAR_ESPACIAL:
    VECINDADES = {"k5": pares_knn(GEO_TABLA, 5), "k3": pares_knn(GEO_TABLA, 3), "k10": pares_knn(GEO_TABLA, 10),
                  "k10p": pares_knn(GEO_TABLA, 10, ponderado=True)}
    if GEO_JSON is not None:
        frontera = pares_frontera(GEO_JSON, VECINDADES["k3"])
        if frontera is not None:
            VECINDADES["frontera"] = frontera
    for nombre, pares in VECINDADES.items():
        print(f"vecindad {nombre:8s}: {pares.groupby('DIVIPOLA').size().mean():.1f} vecinos en promedio")
    train, test_pub, test_priv = (d.merge(GEO_TABLA, on="DIVIPOLA", how="left") for d in (train, test_pub, test_priv))

VEC_COLS = {"hist_tasa_mun": "tasa", "hist_tasa_mes": "tasa_mes", "hist_mag_mun": "mag", "hist_reciente_12": "reciente"}
MESES = pd.DataFrame({"mes": range(1, 13)})


def agregar_vecindad(hist, filas):
    if not USAR_ESPACIAL:
        return filas
    grilla = GEO_TABLA[["DIVIPOLA"]].merge(MESES, how="cross")
    t = historial_congelado(hist, grilla)[["DIVIPOLA", "mes"] + list(VEC_COLS)].rename(columns={"DIVIPOLA": "vecino"})
    salida = filas
    for nombre, pares in VECINDADES.items():
        v = pares.merge(t, on="vecino")
        for c in VEC_COLS:
            v[c] = v[c] * v["peso"]
        g = v.groupby(["DIVIPOLA", "mes"])[list(VEC_COLS) + ["peso"]].sum()
        g = g[list(VEC_COLS)].div(g["peso"], axis=0)
        g.columns = [f"vec_{nombre}_{VEC_COLS[c]}" for c in VEC_COLS]
        salida = salida.merge(g.reset_index(), on=["DIVIPOLA", "mes"], how="left")
    # Suavizado espacial: tasa del municipio encogida hacia la de sus 5 vecinos (en lugar del departamento)
    conteo = hist.groupby("DIVIPOLA")["tiene_evento"].agg(["sum", "size"])
    e = salida["DIVIPOLA"].map(conteo["sum"]).fillna(0).to_numpy()
    n = salida["DIVIPOLA"].map(conteo["size"]).fillna(0).to_numpy()
    previa = salida["vec_k5_tasa"].fillna(hist["tiene_evento"].mean()).to_numpy()
    salida["hist_tasa_mun_espacial"] = (e + K["mun"] * previa) / (n + K["mun"])
    return salida


def historial_completo(hist, filas):
    return agregar_vecindad(hist, historial_congelado(hist, filas))


def historial_entrenamiento_completo(hist):
    anios = sorted(hist["anio"].unique())
    return pd.concat([historial_completo(hist[hist.anio < a], hist[hist.anio == a]) for a in anios[1:]],
                     ignore_index=True)

vecindad k5      : 5.0 vecinos en promedio
vecindad k3      : 3.0 vecinos en promedio
vecindad k10     : 10.0 vecinos en promedio
vecindad k10p    : 10.0 vecinos en promedio
vecindad frontera: 5.9 vecinos en promedio


## 2. Lluvia mensual por municipio (CHIRPS) y transformaciones
1. Se busca `lluvia_chirps_mensual.csv` (carpeta actual, `../data/externos/procesados/` o el repo en Drive).
2. Si no está, se descarga de CHIRPS: solo la ventana de Colombia de los 96 meses del reto (unos 100–150 MB en lecturas
   parciales, 1–2 minutos; instala `rasterio` si falta).
3. Se valida: solo meses del reto, sin duplicados, rango físico y que cubra **todas** las filas municipio-mes del reto.
   Si algo falla, las variantes con lluvia se omiten y el notebook sigue con las referencias.

In [7]:
import subprocess
import sys

LLUVIA_CSV = "lluvia_chirps_mensual.csv"
LLUVIA_CANDIDATAS = [LLUVIA_CSV, os.path.join("..", "data", "externos", "procesados", LLUVIA_CSV),
                     "/content/drive/MyDrive/Documentos Juan Felipe/Hackaton_PENUD/data/externos/procesados/" + LLUVIA_CSV]
CHIRPS_URL = "https://data.chc.ucsb.edu/products/CHIRPS-2.0/global_monthly/cogs/chirps-v2.0.{a}.{m:02d}.cog"
MESES_RETO = pd.date_range("2018-01-01", "2025-12-01", freq="MS")   # solo los meses del reto
CLIMA_FIN = pd.Timestamp("2022-09-01")                               # climatología: solo el periodo de entrenamiento


def descargar_lluvia(geojson, hilos=8):
    """Lee de CHIRPS solo la ventana de Colombia de cada mes del reto y promedia los pixeles de cada municipio."""
    try:
        import rasterio
    except ImportError:
        subprocess.run([sys.executable, "-m", "pip", "install", "-q", "rasterio"], check=True)
        import rasterio
    from concurrent.futures import ThreadPoolExecutor

    import shapely
    from rasterio.windows import Window
    from shapely.geometry import shape

    os.environ.setdefault("GDAL_DISABLE_READDIR_ON_OPEN", "EMPTY_DIR")
    feats = [f for f in geojson["features"] if f.get("geometry")]
    codigos = [str(f["properties"]["cod_dane_mpio"]).zfill(5) for f in feats]
    geoms = [shape(f["geometry"]) for f in feats]
    b = np.array([g.bounds for g in geoms])
    oeste, sur, este, norte = b[:, 0].min() - 0.1, b[:, 1].min() - 0.1, b[:, 2].max() + 0.1, b[:, 3].max() + 0.1

    def leer(fecha):
        with rasterio.open("/vsicurl/" + CHIRPS_URL.format(a=fecha.year, m=fecha.month)) as src:
            f0, c0 = src.index(oeste, norte)
            f1, c1 = src.index(este, sur)
            ventana = Window(c0, f0, c1 - c0 + 1, f1 - f0 + 1)
            matriz = src.read(1, window=ventana).astype("float64")
            if src.nodata is not None:
                matriz[matriz == src.nodata] = np.nan
            matriz[matriz < 0] = np.nan
            return matriz, src.window_transform(ventana)

    with ThreadPoolExecutor(hilos) as ex:
        res = list(ex.map(leer, MESES_RETO))
    pila, tr = np.stack([r[0] for r in res]), res[0][1]
    _, alto, ancho = pila.shape
    validos = ~np.isnan(pila).any(axis=0)
    fv, cv = np.nonzero(validos)
    xs = tr.c + (np.arange(ancho) + 0.5) * tr.a
    ys = tr.f + (np.arange(alto) + 0.5) * tr.e
    plana = pila.reshape(len(MESES_RETO), -1)
    columnas = []
    for g in geoms:
        minx, miny, maxx, maxy = g.bounds
        cc, ff = np.meshgrid(np.nonzero((xs >= minx) & (xs <= maxx))[0], np.nonzero((ys >= miny) & (ys <= maxy))[0])
        sel = np.array([], dtype=int)
        if cc.size:
            dentro = shapely.contains_xy(g, xs[cc], ys[ff])
            sel = ff[dentro] * ancho + cc[dentro]
            sel = sel[validos.ravel()[sel]]
        if len(sel) == 0:   # municipio pequeño: el pixel válido más cercano a su punto representativo
            p = g.representative_point()
            k = int(np.argmin((xs[cv] - p.x) ** 2 + (ys[fv] - p.y) ** 2))
            sel = np.array([fv[k] * ancho + cv[k]])
        columnas.append(plana[:, sel].mean(axis=1))
    return pd.DataFrame({"DIVIPOLA": np.tile(codigos, len(MESES_RETO)),
                         "fecha": np.repeat(MESES_RETO, len(codigos)),
                         "lluvia_mm": np.column_stack(columnas).ravel()})


def cargar_lluvia():
    for ruta in LLUVIA_CANDIDATAS:
        if os.path.exists(ruta):
            t = pd.read_csv(ruta, dtype={"DIVIPOLA": str}, parse_dates=["fecha"])
            return t.assign(DIVIPOLA=t["DIVIPOLA"].str.zfill(5))[["DIVIPOLA", "fecha", "lluvia_mm"]], f"archivo ({ruta})"
    if GEO_JSON is None:
        return None, "no disponible (no hay polígonos para calcularla)"
    try:
        t = descargar_lluvia(GEO_JSON)
        t.to_csv(LLUVIA_CSV, index=False)
        return t, "descargada de CHIRPS (UC Santa Barbara)"
    except Exception as error:
        print(f"⚠ No se pudo descargar la lluvia: {error}")
        return None, "no disponible"


def validar_lluvia(t, llaves):
    cruce = llaves.merge(t, on=["DIVIPOLA", "fecha"], how="left")
    pruebas = [
        ("solo meses del reto (2018-01 → 2025-12)", t.fecha.min() >= MESES_RETO[0] and t.fecha.max() <= MESES_RETO[-1]),
        ("sin llaves duplicadas", not t.duplicated(["DIVIPOLA", "fecha"]).any()),
        ("lluvia entre 0 y 3,000 mm/mes", bool(t.lluvia_mm.between(0, 3000).all())),
        ("96 meses por municipio (para los acumulados y rezagos)", bool((t.groupby("DIVIPOLA").size() == 96).all())),
        (f"cubre todas las filas municipio-mes del reto (faltan {int(cruce.lluvia_mm.isna().sum())})",
         bool(cruce.lluvia_mm.notna().all())),
    ]
    for texto, ok in pruebas:
        print(("✅ " if ok else "❌ ") + texto)
    return all(ok for _, ok in pruebas)


def variables_lluvia(t):
    t = t.sort_values(["DIVIPOLA", "fecha"]).reset_index(drop=True)
    t["mes"] = t["fecha"].dt.month
    g = t.groupby("DIVIPOLA")["lluvia_mm"]
    t["lluvia_log"] = np.log1p(t["lluvia_mm"])
    t["lluvia_3m_log"] = np.log1p(g.transform(lambda s: s.rolling(3, min_periods=3).sum()))
    clima = (t[t["fecha"] <= CLIMA_FIN].groupby(["DIVIPOLA", "mes"])[["lluvia_log", "lluvia_3m_log"]].mean()
             .rename(columns=lambda c: c + "_clima"))
    t = t.join(clima, on=["DIVIPOLA", "mes"])
    t["lluvia_anom"] = t["lluvia_log"] - t["lluvia_log_clima"]
    t["lluvia_3m_anom"] = t["lluvia_3m_log"] - t["lluvia_3m_log_clima"]
    por_mun = t.groupby("DIVIPOLA")
    t["lluvia_anom_rezago1"] = por_mun["lluvia_anom"].shift(1)
    # Placebo: las mismas variables, pero del mismo mes del año anterior
    t["placebo_anom"] = por_mun["lluvia_anom"].shift(12)
    t["placebo_3m_anom"] = por_mun["lluvia_3m_anom"].shift(12)
    t["placebo_anom_rezago1"] = por_mun["lluvia_anom"].shift(13)
    return t[["DIVIPOLA", "fecha", "lluvia_log", "lluvia_3m_log", "lluvia_anom", "lluvia_3m_anom",
              "lluvia_anom_rezago1", "placebo_anom", "placebo_3m_anom", "placebo_anom_rezago1"]]


LLUVIA, origen_lluvia = cargar_lluvia()
llaves_reto = pd.concat([train, test_pub, test_priv])[["DIVIPOLA", "fecha"]].drop_duplicates()
USAR_LLUVIA = LLUVIA is not None and validar_lluvia(LLUVIA, llaves_reto)
if USAR_LLUVIA:
    LLUVIA_VARS = variables_lluvia(LLUVIA)
    train, test_pub, test_priv = (d.merge(LLUVIA_VARS, on=["DIVIPOLA", "fecha"], how="left")
                                  for d in (train, test_pub, test_priv))
    resumen_ll = LLUVIA_VARS.assign(anio=LLUVIA_VARS.fecha.dt.year).groupby("anio")["lluvia_anom"].mean()
    print("Anomalía media nacional por año (log; > 0 = más lluvioso de lo normal):", resumen_ll.round(3).to_dict())
print(f"Lluvia: {origen_lluvia} → variantes con lluvia {'ACTIVADAS' if USAR_LLUVIA else 'DESACTIVADAS'}")

✅ solo meses del reto (2018-01 → 2025-12)
✅ sin llaves duplicadas
✅ lluvia entre 0 y 3,000 mm/mes
✅ 96 meses por municipio (para los acumulados y rezagos)
✅ cubre todas las filas municipio-mes del reto (faltan 0)
Anomalía media nacional por año (log; > 0 = más lluvioso de lo normal): {2018: -0.062, 2019: -0.083, 2020: -0.067, 2021: 0.041, 2022: 0.226, 2023: 0.034, 2024: -0.022, 2025: 0.151}
Lluvia: archivo (..\data\externos\procesados\lluvia_chirps_mensual.csv) → variantes con lluvia ACTIVADAS


## 3. Variables (las de la ronda 3 + grupos de lluvia)

In [8]:
ONI_SERIE = pd.concat([train, test_pub, test_priv])[["fecha", "ONI"]].groupby("fecha")["ONI"].median()


def features_base(df):
    X = pd.DataFrame(index=df.index)
    for c in COV_ANUALES:
        X["log_" + c] = np.log1p(df[c].clip(lower=0))
    X["ingresos_rango_anio"] = df.groupby("anio")["ingresos_tributarios"].rank(pct=True)
    for c in COV_ESTATICAS + ["ONI"] + HIST_COLS:
        X[c] = df[c]
    for c in [c for c in df.columns if c.endswith("_imputado")]:
        X[c] = df[c]
    for k in (1, 2, 3):
        X[f"ONI_rezago_{k}"] = (df["fecha"] - pd.DateOffset(months=k)).map(ONI_SERIE).to_numpy()
    X["mes"] = df["mes"]
    X["mes_sin"], X["mes_cos"] = np.sin(2 * np.pi * df["mes"] / 12), np.cos(2 * np.pi * df["mes"] / 12)
    X["departamento"] = df["DIVIPOLA"].str[:2].astype(int)
    X["oni_x_inundacion"] = df["ONI"] * df["hist_pct_inund"]
    X["oni_x_tasa"] = df["ONI"] * df["hist_tasa_mun"]
    return X


def features(df, cfg):
    X = features_base(df)
    if cfg.get("espacial"):
        grupos, v = cfg["grupos"], cfg["vecindad"]
        if "ubicacion" in grupos:
            X["lat"], X["lon"] = df["lat"], df["lon"]
        if "poblacion_area" in grupos:
            X["log_area"] = np.log1p(df["area_km2"])
            X["log_poblacion"] = np.log1p(df["densidad_poblacional"].clip(lower=0) * df["area_km2"])
        if "vecinos" in grupos:
            X["hist_vecinos_tasa"], X["hist_vecinos_tasa_mes"] = df[f"vec_{v}_tasa"], df[f"vec_{v}_tasa_mes"]
            if cfg.get("vecinos_plus"):
                X["hist_vecinos_mag"], X["hist_vecinos_reciente"] = df[f"vec_{v}_mag"], df[f"vec_{v}_reciente"]
        if cfg.get("suavizado_espacial"):
            X["hist_tasa_mun_espacial"] = df["hist_tasa_mun_espacial"]
    return X

In [9]:
if "features_sin_lluvia" not in globals():     # evita envolver dos veces si se re-ejecuta la celda
    features_sin_lluvia = features
LLUVIA_ANOM = ["lluvia_anom", "lluvia_3m_anom", "lluvia_anom_rezago1"]


def features(df, cfg):
    X = features_sin_lluvia(df, cfg)
    grupos = cfg.get("lluvia", ())
    if "nivel" in grupos:
        X["lluvia_log"], X["lluvia_3m_log"] = df["lluvia_log"], df["lluvia_3m_log"]
    if "anomalia" in grupos:
        for c in LLUVIA_ANOM:
            X[c] = df[c]
        X["lluvia_anom_x_tasa"] = df["lluvia_anom"] * df["hist_tasa_mes"]
    if "placebo" in grupos:     # mismas columnas, pero con la lluvia de hace 12 meses
        for c in LLUVIA_ANOM:
            X[c] = df[c.replace("lluvia_", "placebo_")]
        X["lluvia_anom_x_tasa"] = df["placebo_anom"] * df["hist_tasa_mes"]
    return X

## 4. Métricas oficiales y validador de la plantilla

In [10]:
from sklearn.metrics import roc_auc_score

def metricas(pred, verdad):
    m = verdad.merge(pred, on=["DIVIPOLA", "fecha"])
    y = np.log1p(m["personas_desplazadas"])
    est = np.log1p(m["personas_desplazadas_estimadas"])
    lo, hi = np.log1p(m["personas_desplazadas_q10"]), np.log1p(m["personas_desplazadas_q90"])
    wink = (hi - lo) + 10 * np.clip(lo - y, 0, None) + 10 * np.clip(y - hi, 0, None)   # alpha = 0.2
    return {"auc": roc_auc_score(m["tiene_evento"], m["prob_evento"]),
            "rmse_log": float(np.sqrt(np.mean((y - est) ** 2))), "winkler": float(wink.mean())}

def puntos(met, nula):
    """Parte automática de la nota (máx. 75 puntos), como la tabla de posiciones."""
    s_auc = max(0, (met["auc"] - 0.5) / 0.5)
    s_rmse = min(1, max(0, 1 - met["rmse_log"] / nula["rmse_log"]))
    s_wink = min(1, max(0, 1 - met["winkler"] / nula["winkler"]))
    return 100 * (0.50 * s_auc + 0.20 * s_rmse + 0.05 * s_wink)


def prediccion_nula(filas):
    return filas[["DIVIPOLA", "fecha"]].assign(prob_evento=0.5, personas_desplazadas_estimadas=0.0,
                                               personas_desplazadas_q10=0.0, personas_desplazadas_q90=0.0)


def winkler_con_factor(pred, verdad, factor):
    m = verdad.merge(pred, on=["DIVIPOLA", "fecha"])
    y, lo = np.log1p(m["personas_desplazadas"]), np.log1p(m["personas_desplazadas_q10"])
    hi = np.maximum(lo, np.log1p(m["personas_desplazadas_q90"]) * factor)
    return float(((hi - lo) + 10 * np.clip(lo - y, 0, None) + 10 * np.clip(y - hi, 0, None)).mean())

In [11]:
COLUMNAS = ["DIVIPOLA", "fecha", "prob_evento", "personas_desplazadas_estimadas",
            "personas_desplazadas_q10", "personas_desplazadas_q90"]

def validar_entrega(e, test_pub=test_pub, test_priv=test_priv):
    errores = []
    faltan = [c for c in COLUMNAS if c not in e.columns]
    if faltan:
        return print("❌ Faltan columnas:", faltan)
    claves = pd.concat([test_pub, test_priv])[["DIVIPOLA", "fecha"]].assign(fecha=lambda d: pd.to_datetime(d.fecha))
    e2 = e.assign(DIVIPOLA=e.DIVIPOLA.astype(str).str.zfill(5), fecha=pd.to_datetime(e.fecha))
    if e2.duplicated(["DIVIPOLA", "fecha"]).any():
        errores.append("hay filas DIVIPOLA-fecha duplicadas")
    faltantes = len(claves.merge(e2, on=["DIVIPOLA", "fecha"], how="left", indicator=True).query("_merge == 'left_only'"))
    if faltantes:
        errores.append(f"faltan {faltantes} filas de las {len(claves)} esperadas")
    num = e2[COLUMNAS[2:]].apply(pd.to_numeric, errors="coerce")
    if num.isna().any().any():
        errores.append("hay valores vacíos o no numéricos")
    if not num.prob_evento.between(0, 1).all():
        errores.append("prob_evento debe estar entre 0 y 1")
    if (num[COLUMNAS[3:]] < 0).any().any():
        errores.append("las columnas de personas deben ser >= 0")
    if (num.personas_desplazadas_q10 > num.personas_desplazadas_q90).any():
        errores.append("personas_desplazadas_q10 debe ser <= personas_desplazadas_q90")
    print("✅ Entrega válida:", len(e), "filas" if not errores else "", *["\n❌ " + x for x in errores])
    return not errores

## 5. Modelos (iguales a las rondas 3–4) y variantes de la ronda 5

In [12]:
import shutil
import time

import lightgbm as lgb

NOMBRE_EQUIPO = "AICOLOMBIA"
LGBM_BASE = dict(learning_rate=0.05, num_leaves=31, min_child_samples=100, colsample_bytree=0.8, subsample=0.8,
                 subsample_freq=1, reg_lambda=1.0, deterministic=True, force_row_wise=True, verbose=-1)
ES_MESES, ES_PACIENCIA, FACTOR_ARBOLES, MAX_ARBOLES = 12, 100, 1.15, 3000

SIMPLE = dict(num_leaves=15, min_child_samples=200)
E10S = dict(espacial=True, grupos=("ubicacion", "poblacion_area", "vecinos"), vecindad="k5", vecinos_plus=False,
            suavizado_espacial=False, arboles_p=100, arboles_mag=None, arboles_q=None, lgbm=SIMPLE, semillas=(42,))
E7 = dict(espacial=False, arboles_p=None, arboles_mag=None, arboles_q=None, lgbm=dict(learning_rate=0.02),
          semillas=(42,))
VARIANTES = {
    "E10S": E10S,
    "E10S_sin_poblacion": {**E10S, "grupos": ("ubicacion", "vecinos")},
    "E10S_sin_ubicacion": {**E10S, "grupos": ("poblacion_area", "vecinos")},
    "E10S_sin_vecinos": {**E10S, "grupos": ("ubicacion", "poblacion_area")},
    "E10S_k3": {**E10S, "vecindad": "k3"},
    "E10S_k10": {**E10S, "vecindad": "k10"},
    "E10S_k10_ponderado": {**E10S, "vecindad": "k10p"},
    "E10S_frontera": {**E10S, "vecindad": "frontera"},
    "E10S_vecinos_plus": {**E10S, "vecinos_plus": True},
    "E10S_suavizado_espacial": {**E10S, "suavizado_espacial": True},
    "E10S_arboles_50": {**E10S, "arboles_p": 50},
    "E10S_arboles_150": {**E10S, "arboles_p": 150},
    "E10S_hojas_7": {**E10S, "lgbm": {**SIMPLE, "num_leaves": 7}},
    "E10S_lento": {**E10S, "arboles_p": 250, "lgbm": {**SIMPLE, "learning_rate": 0.02}},
    "E10S_todo_simple": {**E10S, "arboles_mag": 100, "arboles_q": 100},
    "E10S_3semillas": {**E10S, "semillas": (42, 7, 2026)},
    "E7_lento": E7,
}
if not USAR_ESPACIAL:
    VARIANTES = {"E7_lento": E7}
elif "frontera" not in VECINDADES:
    VARIANTES.pop("E10S_frontera")

DESCRIPCION = {
    "E10S": "Referencia: E10_espacial_simple de la ronda 2",
    "E10S_sin_poblacion": "E10S sin población ni área",
    "E10S_sin_ubicacion": "E10S sin latitud ni longitud",
    "E10S_sin_vecinos": "E10S sin la tasa de los vecinos",
    "E10S_k3": "Vecinos: 3 más cercanos",
    "E10S_k10": "Vecinos: 10 más cercanos",
    "E10S_k10_ponderado": "Vecinos: 10 más cercanos ponderados por distancia",
    "E10S_frontera": "Vecinos: los que comparten frontera",
    "E10S_vecinos_plus": "Agrega magnitud y tasa reciente de los vecinos",
    "E10S_suavizado_espacial": "Agrega la tasa del municipio encogida hacia sus vecinos",
    "E10S_arboles_50": "Clasificador con 50 árboles",
    "E10S_arboles_150": "Clasificador con 150 árboles",
    "E10S_hojas_7": "Árboles de 7 hojas",
    "E10S_lento": "Tasa de aprendizaje 0.02 con 250 árboles",
    "E10S_todo_simple": "Magnitud y cuantiles también simples (100 árboles)",
    "E10S_3semillas": "Promedio de 3 semillas",
    "E7_lento": "Referencia sin espaciales: E7_lento de la ronda 2",
    "ENS_E10S_E7": "Promedio fijo de E10S y E7_lento",
}


def ajustar(X, y, fechas, clase, cfg, n_arboles, semilla, **extra):
    params = {**LGBM_BASE, **cfg["lgbm"], **extra, "random_state": semilla}
    if n_arboles is not None:
        return clase(**params, n_estimators=n_arboles).fit(X, y)
    meses = np.sort(fechas.unique())
    corte = meses[-min(ES_MESES, max(3, len(meses) // 4))]
    a, b = (fechas < corte).to_numpy(), (fechas >= corte).to_numpy()
    m = clase(**params, n_estimators=MAX_ARBOLES).fit(X[a], y[a], eval_set=[(X[b], y[b])],
                                                     callbacks=[lgb.early_stopping(ES_PACIENCIA, verbose=False)])
    n = max(50, int(round((m.best_iteration_ or MAX_ARBOLES) * FACTOR_ARBOLES)))
    return clase(**params, n_estimators=n).fit(X, y)


def entrenar(filas, cfg):
    X, fechas = features(filas, cfg), filas["fecha"]
    y_ev, y_log = filas["tiene_evento"], np.log1p(filas["personas_desplazadas"])
    conjuntos = []
    for s in cfg["semillas"]:
        m = {"p": ajustar(X, y_ev, fechas, lgb.LGBMClassifier, cfg, cfg["arboles_p"], s, objective="binary", metric="auc"),
             "e": ajustar(X, y_log, fechas, lgb.LGBMRegressor, cfg, cfg["arboles_mag"], s, objective="regression", metric="l2")}
        for q in (0.1, 0.9):
            m[q] = ajustar(X, y_log, fechas, lgb.LGBMRegressor, cfg, cfg["arboles_q"], s, objective="quantile",
                           alpha=q, metric="quantile")
        conjuntos.append(m)
    return conjuntos, None


def predecir_log(conjuntos, filas, cfg):
    X = features(filas, cfg)
    p = filas[["DIVIPOLA", "fecha"]].copy()
    p["prob"] = np.mean([m["p"].predict_proba(X)[:, 1] for m in conjuntos], axis=0)
    p["mag"] = np.mean([np.clip(m["e"].predict(X), 0, None) for m in conjuntos], axis=0)
    p["lo"] = np.mean([np.clip(m[0.1].predict(X), 0, None) for m in conjuntos], axis=0)
    p["hi"] = np.mean([np.clip(m[0.9].predict(X), 0, None) for m in conjuntos], axis=0)
    return p


def a_entrega(cruda, factor_q90=1.0):
    p = cruda[["DIVIPOLA", "fecha"]].copy()
    p["prob_evento"] = cruda["prob"].to_numpy()
    p["personas_desplazadas_estimadas"] = np.expm1(cruda["mag"].to_numpy())
    hi = cruda["hi"].to_numpy() * factor_q90
    p["personas_desplazadas_q10"] = np.expm1(np.minimum(cruda["lo"].to_numpy(), hi))
    p["personas_desplazadas_q90"] = np.expm1(np.maximum(cruda["lo"].to_numpy(), hi))
    return p

In [13]:
VARIANTES = {"E10S": E10S, "E10S_arboles_50": {**E10S, "arboles_p": 50}, "E7_lento": E7}
if USAR_LLUVIA:
    LL = ("nivel", "anomalia")
    VARIANTES.update({
        "E10S_LL": {**E10S, "lluvia": LL},
        "E10S_LL_anom": {**E10S, "lluvia": ("anomalia",)},
        "E10S_LL_50": {**E10S, "lluvia": LL, "arboles_p": 50},
        "E10S_LL_150": {**E10S, "lluvia": LL, "arboles_p": 150},
        "E10S_LL_placebo": {**E10S, "lluvia": ("placebo",)},
        "E7_LL": {**E7, "lluvia": LL},
    })
if not USAR_ESPACIAL:
    VARIANTES = {k: v for k, v in VARIANTES.items() if not v.get("espacial")}

DESCRIPCION.update({
    "E10S": "Referencia sin lluvia: E10S de las rondas 3-4",
    "E10S_arboles_50": "Referencia sin lluvia: E10S con 50 árboles",
    "E7_lento": "Referencia sin lluvia ni espaciales: E7_lento",
    "E10S_LL": "E10S + lluvia (nivel y anomalías) en los 4 modelos",
    "E10S_LL_anom": "E10S + solo anomalías de lluvia",
    "E10S_LL_50": "E10S_LL con 50 árboles en el clasificador",
    "E10S_LL_150": "E10S_LL con 150 árboles en el clasificador",
    "E10S_LL_placebo": "Placebo: anomalías de lluvia del mismo mes del año anterior",
    "E7_LL": "E7_lento + lluvia (nivel y anomalías)",
    "E10S_LL_solo_clasif": "Probabilidad de E10S_LL; magnitud y cuantiles de E10S",
    "E10S_LL_solo_magnitud": "Probabilidad de E10S; magnitud y cuantiles de E10S_LL",
    "CB_d4_LL": "CatBoost prof. 4 con lluvia (clasificador); magnitud y cuantiles de E10S_LL",
    "ENS_E10S_E7": "Referencia: promedio fijo de E10S y E7_lento",
    "ENS_E10S_E7_LL": "Promedio fijo de E10S_LL y E7_LL",
    "ENS_E10S_E7_CB_LL": "Promedio fijo de E10S_LL, E7_LL y CB_d4_LL",
})
NOTAS = {k: "usa lluvia CHIRPS 2018-2025 (entregar solo con confirmación de los organizadores)"
         for k in DESCRIPCION if "_LL" in k}
print("Variantes a entrenar:", list(VARIANTES))

Variantes a entrenar: ['E10S', 'E10S_arboles_50', 'E7_lento', 'E10S_LL', 'E10S_LL_anom', 'E10S_LL_50', 'E10S_LL_150', 'E10S_LL_placebo', 'E7_LL']


## 6. Correr las variantes
El historial y la vecindad se calculan una vez por corte. Por variante: validación F1–F3, factor de `q90`, modelo final
y CSV validado en `entregas_ronda5/`.

In [14]:
CORTES = [("F1", "2020-09-01", "2020-10-01", "2021-09-01"),
          ("F2", "2021-09-01", "2021-10-01", "2022-09-01"),
          ("F3", "2020-09-01", "2021-10-01", "2022-09-01")]
FACTORES = [0.7, 0.8, 0.9, 1.0, 1.1, 1.25, 1.4]
CARPETA = "entregas_ronda5"

t0 = time.time()
PREP = {}
for nombre, fin_hist, ini_val, fin_val in CORTES:
    hist = train[train.fecha <= fin_hist]
    va = train[(train.fecha >= ini_val) & (train.fecha <= fin_val)]
    PREP[nombre] = (historial_entrenamiento_completo(hist), historial_completo(hist, va), va)
PRUEBA = pd.concat([test_pub, test_priv], ignore_index=True)
FINAL = (historial_entrenamiento_completo(train), historial_completo(train, PRUEBA))
print(f"Historial y vecindad preparados en {time.time() - t0:.0f}s")


def evaluar(crudas_por_corte):
    """Métricas por corte, factor de q90 y cobertura a partir de predicciones crudas."""
    por_corte, factores = {}, []
    for corte, cruda in crudas_por_corte.items():
        va = PREP[corte][2]
        pred = a_entrega(cruda)
        met, nula = metricas(pred, va), metricas(prediccion_nula(va), va)
        winklers = {f: winkler_con_factor(pred, va, f) for f in FACTORES}
        factores.append(min(winklers, key=winklers.get))
        m = va.merge(pred, on=["DIVIPOLA", "fecha"])
        cobertura = float(((m.personas_desplazadas >= m.personas_desplazadas_q10) &
                           (m.personas_desplazadas <= m.personas_desplazadas_q90)).mean())
        por_corte[corte] = {**met, "puntos": puntos(met, nula), "cobertura": cobertura}
    return pd.DataFrame(por_corte).T, float(np.median(factores))


def guardar(variante, cruda_final, factor):
    entrega = a_entrega(cruda_final, factor)
    entrega["fecha"] = pd.to_datetime(entrega["fecha"]).dt.strftime("%Y-%m-%d")
    entrega = entrega[COLUMNAS]
    if not validar_entrega(entrega):
        return "INVÁLIDA"
    archivo = f"{CARPETA}/entrega_{NOMBRE_EQUIPO}_{variante}.csv"
    entrega.to_csv(archivo, index=False)
    return archivo


os.makedirs(CARPETA, exist_ok=True)
CRUDAS, CRUDAS_FINAL, resumen, detalle = {}, {}, [], []


def registrar(variante, d, factor, archivo, segundos):
    resumen.append({"ronda": 5, "variante": variante, "descripcion": DESCRIPCION[variante],
                    "puntos_prom": d.puntos.mean(), "puntos_desv": d.puntos.std(),
                    "puntos_F1": d.loc["F1", "puntos"], "puntos_F2": d.loc["F2", "puntos"],
                    "puntos_F3": d.loc["F3", "puntos"], "auc_prom": d.auc.mean(), "rmse_log_prom": d.rmse_log.mean(),
                    "winkler_prom": d.winkler.mean(), "cobertura_prom": d.cobertura.mean(), "factor_q90": factor,
                    "archivo": archivo})
    for corte in d.index:
        detalle.append({"ronda": 5, "variante": variante, "corte": corte, **d.loc[corte].to_dict()})
    print(f"{variante:22s} puntos {d.puntos.mean():5.1f} ± {d.puntos.std():.1f} · F2 {d.loc['F2', 'puntos']:5.1f} "
          f"· F3 {d.loc['F3', 'puntos']:5.1f} · {segundos:.0f}s")


for variante, cambios in VARIANTES.items():
    t1 = time.time()
    cfg = cambios
    CRUDAS[variante] = {}
    for corte, (filas_hist, filas_va, va) in PREP.items():
        modelos_cv, _ = entrenar(filas_hist, cfg)
        CRUDAS[variante][corte] = predecir_log(modelos_cv, filas_va, cfg)
    d, factor = evaluar(CRUDAS[variante])
    modelos_fin, _ = entrenar(FINAL[0], cfg)
    CRUDAS_FINAL[variante] = predecir_log(modelos_fin, FINAL[1], cfg)
    registrar(variante, d, factor, guardar(variante, CRUDAS_FINAL[variante], factor), time.time() - t1)

print(f"Modelos LightGBM listos en {(time.time() - t0) / 60:.1f} min")

Historial y vecindad preparados en 6s


c:\Users\ingju\AppData\Local\Programs\Python\Python312\Lib\site-packages\lightgbm\sklearn.py:1106: LGBMDeprecationWarning: The argument 'eval_set' is deprecated, use 'eval_X' and 'eval_y' instead.
  eval_set = _validate_eval_set_Xy(eval_set=eval_set, eval_X=eval_X, eval_y=eval_y)
c:\Users\ingju\AppData\Local\Programs\Python\Python312\Lib\site-packages\lightgbm\sklearn.py:1106: LGBMDeprecationWarning: The argument 'eval_set' is deprecated, use 'eval_X' and 'eval_y' instead.
  eval_set = _validate_eval_set_Xy(eval_set=eval_set, eval_X=eval_X, eval_y=eval_y)
c:\Users\ingju\AppData\Local\Programs\Python\Python312\Lib\site-packages\lightgbm\sklearn.py:1106: LGBMDeprecationWarning: The argument 'eval_set' is deprecated, use 'eval_X' and 'eval_y' instead.
  eval_set = _validate_eval_set_Xy(eval_set=eval_set, eval_X=eval_X, eval_y=eval_y)
c:\Users\ingju\AppData\Local\Programs\Python\Python312\Lib\site-packages\lightgbm\sklearn.py:1106: LGBMDeprecationWarning: The argument 'eval_set' is depreca

✅ Entrega válida: 42978 filas
E10S                   puntos  22.0 ± 2.5 · F2  24.6 · F3  19.6 · 10s


c:\Users\ingju\AppData\Local\Programs\Python\Python312\Lib\site-packages\lightgbm\sklearn.py:1106: LGBMDeprecationWarning: The argument 'eval_set' is deprecated, use 'eval_X' and 'eval_y' instead.
  eval_set = _validate_eval_set_Xy(eval_set=eval_set, eval_X=eval_X, eval_y=eval_y)
c:\Users\ingju\AppData\Local\Programs\Python\Python312\Lib\site-packages\lightgbm\sklearn.py:1106: LGBMDeprecationWarning: The argument 'eval_set' is deprecated, use 'eval_X' and 'eval_y' instead.
  eval_set = _validate_eval_set_Xy(eval_set=eval_set, eval_X=eval_X, eval_y=eval_y)
c:\Users\ingju\AppData\Local\Programs\Python\Python312\Lib\site-packages\lightgbm\sklearn.py:1106: LGBMDeprecationWarning: The argument 'eval_set' is deprecated, use 'eval_X' and 'eval_y' instead.
  eval_set = _validate_eval_set_Xy(eval_set=eval_set, eval_X=eval_X, eval_y=eval_y)
c:\Users\ingju\AppData\Local\Programs\Python\Python312\Lib\site-packages\lightgbm\sklearn.py:1106: LGBMDeprecationWarning: The argument 'eval_set' is depreca

✅ Entrega válida: 42978 filas
E10S_arboles_50        puntos  22.1 ± 2.7 · F2  25.1 · F3  19.8 · 8s


c:\Users\ingju\AppData\Local\Programs\Python\Python312\Lib\site-packages\lightgbm\sklearn.py:1106: LGBMDeprecationWarning: The argument 'eval_set' is deprecated, use 'eval_X' and 'eval_y' instead.
  eval_set = _validate_eval_set_Xy(eval_set=eval_set, eval_X=eval_X, eval_y=eval_y)
c:\Users\ingju\AppData\Local\Programs\Python\Python312\Lib\site-packages\lightgbm\sklearn.py:1106: LGBMDeprecationWarning: The argument 'eval_set' is deprecated, use 'eval_X' and 'eval_y' instead.
  eval_set = _validate_eval_set_Xy(eval_set=eval_set, eval_X=eval_X, eval_y=eval_y)
c:\Users\ingju\AppData\Local\Programs\Python\Python312\Lib\site-packages\lightgbm\sklearn.py:1106: LGBMDeprecationWarning: The argument 'eval_set' is deprecated, use 'eval_X' and 'eval_y' instead.
  eval_set = _validate_eval_set_Xy(eval_set=eval_set, eval_X=eval_X, eval_y=eval_y)
c:\Users\ingju\AppData\Local\Programs\Python\Python312\Lib\site-packages\lightgbm\sklearn.py:1106: LGBMDeprecationWarning: The argument 'eval_set' is depreca

... (9 outputs left out)

c:\Users\ingju\AppData\Local\Programs\Python\Python312\Lib\site-packages\lightgbm\sklearn.py:1106: LGBMDeprecationWarning: The argument 'eval_set' is deprecated, use 'eval_X' and 'eval_y' instead.
  eval_set = _validate_eval_set_Xy(eval_set=eval_set, eval_X=eval_X, eval_y=eval_y)
c:\Users\ingju\AppData\Local\Programs\Python\Python312\Lib\site-packages\lightgbm\sklearn.py:1106: LGBMDeprecationWarning: The argument 'eval_set' is deprecated, use 'eval_X' and 'eval_y' instead.
  eval_set = _validate_eval_set_Xy(eval_set=eval_set, eval_X=eval_X, eval_y=eval_y)
c:\Users\ingju\AppData\Local\Programs\Python\Python312\Lib\site-packages\lightgbm\sklearn.py:1106: LGBMDeprecationWarning: The argument 'eval_set' is deprecated, use 'eval_X' and 'eval_y' instead.
  eval_set = _validate_eval_set_Xy(eval_set=eval_set, eval_X=eval_X, eval_y=eval_y)
c:\Users\ingju\AppData\Local\Programs\Python\Python312\Lib\site-packages\lightgbm\sklearn.py:1106: LGBMDeprecationWarning: The argument 'eval_set' is depreca

✅ Entrega válida: 42978 filas
E10S_LL_placebo        puntos  21.9 ± 2.4 · F2  24.3 · F3  19.6 · 8s


c:\Users\ingju\AppData\Local\Programs\Python\Python312\Lib\site-packages\lightgbm\sklearn.py:1106: LGBMDeprecationWarning: The argument 'eval_set' is deprecated, use 'eval_X' and 'eval_y' instead.
  eval_set = _validate_eval_set_Xy(eval_set=eval_set, eval_X=eval_X, eval_y=eval_y)
c:\Users\ingju\AppData\Local\Programs\Python\Python312\Lib\site-packages\lightgbm\sklearn.py:1106: LGBMDeprecationWarning: The argument 'eval_set' is deprecated, use 'eval_X' and 'eval_y' instead.
  eval_set = _validate_eval_set_Xy(eval_set=eval_set, eval_X=eval_X, eval_y=eval_y)
c:\Users\ingju\AppData\Local\Programs\Python\Python312\Lib\site-packages\lightgbm\sklearn.py:1106: LGBMDeprecationWarning: The argument 'eval_set' is deprecated, use 'eval_X' and 'eval_y' instead.
  eval_set = _validate_eval_set_Xy(eval_set=eval_set, eval_X=eval_X, eval_y=eval_y)
c:\Users\ingju\AppData\Local\Programs\Python\Python312\Lib\site-packages\lightgbm\sklearn.py:1106: LGBMDeprecationWarning: The argument 'eval_set' is depreca

✅ Entrega válida: 42978 filas
E7_LL                  puntos  25.4 ± 2.6 · F2  27.2 · F3  22.4 · 24s
Modelos LightGBM listos en 2.1 min


## 7. CatBoost con lluvia, combinaciones y ensambles fijos
* **Combinaciones:** los 4 modelos (probabilidad, magnitud, q10 y q90) se entrenan por separado, así que tomar la
  probabilidad de una variante y la magnitud de otra equivale a usar la lluvia solo en una parte.
* **Ensambles:** promedio de probabilidad, magnitud y cuantiles (en log1p) de los miembros.

In [15]:
PROBAR_CATBOOST = True
if PROBAR_CATBOOST and "E10S_LL" in CRUDAS:
    try:
        from catboost import CatBoostClassifier
    except ImportError:
        subprocess.run([sys.executable, "-m", "pip", "install", "-q", "catboost"], check=True)
        from catboost import CatBoostClassifier
    CFG_CB = VARIANTES["E10S_LL"]
    t1 = time.time()
    entrenar_cb = lambda filas: CatBoostClassifier(iterations=300, learning_rate=0.05, depth=4, l2_leaf_reg=3,
                                                   random_seed=42, verbose=0, allow_writing_files=False,
                                                   thread_count=-1).fit(features(filas, CFG_CB), filas["tiene_evento"])
    CRUDAS["CB_d4_LL"] = {}
    for corte, (filas_hist, filas_va, va) in PREP.items():
        prob = entrenar_cb(filas_hist).predict_proba(features(filas_va, CFG_CB))[:, 1]
        CRUDAS["CB_d4_LL"][corte] = CRUDAS["E10S_LL"][corte].assign(prob=prob)
    d, factor = evaluar(CRUDAS["CB_d4_LL"])
    prob = entrenar_cb(FINAL[0]).predict_proba(features(FINAL[1], CFG_CB))[:, 1]
    CRUDAS_FINAL["CB_d4_LL"] = CRUDAS_FINAL["E10S_LL"].assign(prob=prob)
    registrar("CB_d4_LL", d, factor, guardar("CB_d4_LL", CRUDAS_FINAL["CB_d4_LL"], factor), time.time() - t1)

COMBINACIONES = {"E10S_LL_solo_clasif": ("E10S_LL", "E10S"),       # (probabilidad de, magnitud y cuantiles de)
                 "E10S_LL_solo_magnitud": ("E10S", "E10S_LL")}
for nombre, (de_prob, de_mag) in COMBINACIONES.items():
    if {de_prob, de_mag} <= set(CRUDAS):
        crudas = {c: CRUDAS[de_mag][c].assign(prob=CRUDAS[de_prob][c]["prob"].to_numpy()) for c in PREP}
        d, factor = evaluar(crudas)
        CRUDAS[nombre] = crudas
        CRUDAS_FINAL[nombre] = CRUDAS_FINAL[de_mag].assign(prob=CRUDAS_FINAL[de_prob]["prob"].to_numpy())
        registrar(nombre, d, factor, guardar(nombre, CRUDAS_FINAL[nombre], factor), 0)

ENSAMBLES = {"ENS_E10S_E7": ["E10S", "E7_lento"], "ENS_E10S_E7_LL": ["E10S_LL", "E7_LL"],
             "ENS_E10S_E7_CB_LL": ["E10S_LL", "E7_LL", "CB_d4_LL"]}
promedio = lambda partes: partes[0][["DIVIPOLA", "fecha"]].assign(
    **{c: np.mean([p[c].to_numpy() for p in partes], axis=0) for c in ["prob", "mag", "lo", "hi"]})
for nombre, miembros in ENSAMBLES.items():
    if set(miembros) <= set(CRUDAS):
        crudas = {corte: promedio([CRUDAS[m][corte] for m in miembros]) for corte in PREP}
        d, factor = evaluar(crudas)
        CRUDAS[nombre] = crudas
        CRUDAS_FINAL[nombre] = promedio([CRUDAS_FINAL[m] for m in miembros])
        registrar(nombre, d, factor, guardar(nombre, CRUDAS_FINAL[nombre], factor), 0)

CONFIGS = {**VARIANTES, "CB_d4_LL": {"catboost": dict(depth=4, iterations=300), "lluvia": ("nivel", "anomalia")},
           **{k: {"combinacion": v} for k, v in COMBINACIONES.items()},
           **{k: {"ensamble": v} for k, v in ENSAMBLES.items()}}
print(f"Total: {(time.time() - t0) / 60:.1f} min")

✅ Entrega válida: 42978 filas
CB_d4_LL               puntos  25.5 ± 2.3 · F2  27.3 · F3  22.8 · 13s
✅ Entrega válida: 42978 filas
E10S_LL_solo_clasif    puntos  25.2 ± 2.3 · F2  26.9 · F3  22.6 · 0s
✅ Entrega válida: 42978 filas
E10S_LL_solo_magnitud  puntos  22.3 ± 2.5 · F2  24.9 · F3  20.0 · 0s
✅ Entrega válida: 42978 filas
ENS_E10S_E7            puntos  22.1 ± 2.6 · F2  24.7 · F3  19.5 · 0s
✅ Entrega válida: 42978 filas
ENS_E10S_E7_LL         puntos  25.7 ± 2.4 · F2  27.4 · F3  22.9 · 0s
✅ Entrega válida: 42978 filas
ENS_E10S_E7_CB_LL      puntos  25.8 ± 2.4 · F2  27.5 · F3  23.1 · 0s
Total: 2.4 min


## 8. Resultados: ronda 5 frente a los mejores de las rondas anteriores

In [16]:
# Mejores resultados anteriores (datos reales, del registro central), por corte
ANTERIORES = pd.DataFrame([
    (3, 'E10S_arboles_50', 0.696, 0.732, 0.681, 1.352, 1.26, 1.263, 3.501, 3.295, 3.45, 21.587, 25.05, 19.767),
    (3, 'ENS_E10S_E7', 0.702, 0.727, 0.679, 1.351, 1.259, 1.265, 3.496, 3.273, 3.457, 22.165, 24.663, 19.458),
    (3, 'E10S_k3', 0.699, 0.73, 0.679, 1.351, 1.26, 1.264, 3.51, 3.299, 3.454, 21.895, 24.868, 19.468),
    (3, 'E10S', 0.697, 0.727, 0.68, 1.352, 1.26, 1.263, 3.501, 3.295, 3.45, 21.716, 24.608, 19.618),
    (3, 'E7_lento', 0.701, 0.723, 0.672, 1.351, 1.26, 1.268, 3.499, 3.262, 3.475, 22.057, 24.199, 18.697),
    (4, 'ENS_E10S_E7_CB', 0.698, 0.732, 0.671, 1.351, 1.259, 1.264, 3.497, 3.279, 3.453, 21.814, 25.15, 18.759),
    (4, 'CB_d4', 0.687, 0.736, 0.656, 1.352, 1.26, 1.263, 3.501, 3.295, 3.45, 20.652, 25.506, 17.24),
    (2, 'E9_ensamble_top3', 0.7, 0.73, 0.677, 1.351, 1.258, 1.265, 3.505, 3.269, 3.459, 21.975, 24.909, 19.242),
    (1, 'V1_early_stopping', 0.679, 0.722, 0.647, 1.353, 1.263, 1.268, 3.513, 3.248, 3.482, 19.872, 24.07, 16.25),
], columns=["ronda", "variante", "auc_F1", "auc_F2", "auc_F3", "rmse_F1", "rmse_F2", "rmse_F3",
            "wink_F1", "wink_F2", "wink_F3", "puntos_F1", "puntos_F2", "puntos_F3"])
P = ["puntos_F1", "puntos_F2", "puntos_F3"]
anteriores = ANTERIORES.assign(descripcion="(ronda anterior)", archivo="zip de esa ronda",
                               puntos_prom=ANTERIORES[P].mean(axis=1), puntos_desv=ANTERIORES[P].std(axis=1),
                               auc_prom=ANTERIORES[["auc_F1", "auc_F2", "auc_F3"]].mean(axis=1),
                               rmse_log_prom=ANTERIORES[["rmse_F1", "rmse_F2", "rmse_F3"]].mean(axis=1),
                               winkler_prom=ANTERIORES[["wink_F1", "wink_F2", "wink_F3"]].mean(axis=1))

r5 = pd.DataFrame(resumen)
for nombre in ["E10S", "E10S_arboles_50", "E7_lento", "ENS_E10S_E7"]:
    if nombre in set(r5.variante) and nombre in set(anteriores.variante):
        a = r5.set_index("variante").loc[nombre, "puntos_prom"]
        b = anteriores[anteriores.variante == nombre]["puntos_prom"].iloc[0]
        print(f"Reproducibilidad {nombre:16s}: antes {b:.2f} vs ronda 5 {a:.2f} "
              + ("✅" if abs(a - b) < 0.05 else "⚠ difiere: normal si cambió el entorno o la versión de LightGBM"))

referencia = "E10S" if "E10S" in set(r5.variante) else "E7_lento"
ref = r5.set_index("variante").loc[referencia]
mejor_anterior = anteriores.sort_values("puntos_prom", ascending=False).iloc[0]
todas = pd.concat([r5, anteriores[~anteriores.variante.isin(r5.variante)]], ignore_index=True)
todas[f"delta_vs_{referencia}"] = todas["puntos_prom"] - ref["puntos_prom"]
todas["delta_vs_mejor_anterior"] = todas["puntos_prom"] - mejor_anterior["puntos_prom"]
todas["delta_auc"] = todas["auc_prom"] - ref["auc_prom"]
todas["mejora_a_la_referencia"] = (todas[f"delta_vs_{referencia}"] >= 0.5) & (todas["puntos_F3"] >= ref["puntos_F3"])
tabla = todas.sort_values("puntos_prom", ascending=False).reset_index(drop=True)
columnas = ["ronda", "variante", "descripcion", "puntos_prom", "puntos_desv", "puntos_F1", "puntos_F2", "puntos_F3",
            f"delta_vs_{referencia}", "delta_vs_mejor_anterior", "auc_prom", "delta_auc", "mejora_a_la_referencia",
            "rmse_log_prom", "winkler_prom", "archivo"]
pd.set_option("display.width", 250)
pd.set_option("display.max_colwidth", 55)
print(f"Mejor anterior: {mejor_anterior.variante} (ronda {mejor_anterior.ronda}) · {mejor_anterior.puntos_prom:.2f} puntos")
display(tabla[columnas].round(3))

print("Detalle por corte (ronda 5):")
display(pd.DataFrame(detalle).pivot(index="variante", columns="corte",
                                    values=["auc", "rmse_log", "winkler", "puntos"]).round(3))

# ¿Aporta la lluvia? Cada variante contra su par sin lluvia
r5i = r5.set_index("variante")
det5 = pd.DataFrame(detalle).set_index(["variante", "corte"])
pares = [("E10S_LL", "E10S", "todas las variables de lluvia"), ("E10S_LL_anom", "E10S", "solo anomalías"),
         ("E10S_LL_placebo", "E10S", "PLACEBO (lluvia de hace 12 meses)"), ("E10S_LL_50", "E10S_arboles_50", "50 árboles"),
         ("E7_LL", "E7_lento", "E7 con lluvia"), ("E10S_LL_solo_clasif", "E10S", "lluvia solo en la probabilidad"),
         ("E10S_LL_solo_magnitud", "E10S", "lluvia solo en magnitud y cuantiles"),
         ("ENS_E10S_E7_LL", "ENS_E10S_E7", "ensamble con lluvia")]
aporte = []
for con, sin, texto in pares:
    if con in r5i.index and sin in r5i.index:
        fila = {"variante": con, "frente_a": sin, "que_mide": texto,
                "delta_puntos": r5i.loc[con, "puntos_prom"] - r5i.loc[sin, "puntos_prom"],
                "delta_auc": r5i.loc[con, "auc_prom"] - r5i.loc[sin, "auc_prom"],
                "delta_rmse_log": r5i.loc[con, "rmse_log_prom"] - r5i.loc[sin, "rmse_log_prom"],
                "delta_winkler": r5i.loc[con, "winkler_prom"] - r5i.loc[sin, "winkler_prom"]}
        for c in ["F1", "F2", "F3"]:
            fila[f"delta_puntos_{c}"] = det5.loc[(con, c), "puntos"] - det5.loc[(sin, c), "puntos"]
        aporte.append(fila)
if aporte:
    print("Aporte de la lluvia (positivo = mejor; en RMSE y Winkler, negativo = mejor):")
    display(pd.DataFrame(aporte).round(3))
    if {"E10S_LL_anom", "E10S_LL_placebo"} <= set(r5i.index):
        real = r5i.loc["E10S_LL_anom", "puntos_prom"] - ref["puntos_prom"]
        placebo = r5i.loc["E10S_LL_placebo", "puntos_prom"] - ref["puntos_prom"]
        print(f"Anomalías reales {real:+.2f} vs placebo {placebo:+.2f} puntos → "
              + ("la ganancia sí viene de la lluvia del mes" if real - placebo >= 0.3 else
                 "no se distingue del placebo: la ganancia (si la hay) no es de la lluvia del mes"))

if "E10S_LL" in VARIANTES:
    imp_modelo = entrenar(FINAL[0], VARIANTES["E10S_LL"])[0][0]["p"]
    imp = pd.Series(imp_modelo.booster_.feature_importance("gain"), index=imp_modelo.feature_name_)
    imp = imp / imp.sum()
    print(f"Importancia de la lluvia en el clasificador de E10S_LL: {imp[imp.index.str.startswith('lluvia')].sum():.1%} del total")
    display(imp.sort_values(ascending=False).head(20).round(3).to_frame("importancia"))

tabla.to_csv(f"{CARPETA}/resultados_ronda5.csv", index=False)
candidatas = tabla[tabla["mejora_a_la_referencia"] & (tabla["ronda"] == 5)]
if len(candidatas):
    mejor = candidatas.iloc[0]
    aviso = " · ⚠ usa lluvia: entregar solo con confirmación de los organizadores" if "_LL" in mejor.variante else ""
    print(f"Mejora a {referencia}: {mejor.variante} ({mejor.descripcion}) · {mejor.puntos_prom:.2f} puntos "
          f"(+{mejor[f'delta_vs_{referencia}']:.2f}) · {mejor.archivo}{aviso}")
else:
    print(f"Ninguna variante mejora ≥ 0.5 puntos a {referencia} sin empeorar F3: mantener {referencia}.")

Reproducibilidad E10S            : antes 21.98 vs ronda 5 21.98 ✅
Reproducibilidad E10S_arboles_50 : antes 22.13 vs ronda 5 22.13 ✅
Reproducibilidad E7_lento        : antes 21.65 vs ronda 5 21.65 ✅
Reproducibilidad ENS_E10S_E7     : antes 22.10 vs ronda 5 22.10 ✅
Mejor anterior: E10S_arboles_50 (ronda 3) · 22.13 puntos


,ronda,variante,descripcion,puntos_prom,puntos_desv,puntos_F1,puntos_F2,puntos_F3,delta_vs_E10S,delta_vs_mejor_anterior,auc_prom,delta_auc,mejora_a_la_referencia,rmse_log_prom,winkler_prom,archivo
0,5,ENS_E10S_E7_CB_LL,"Promedio fijo de E10S_LL, E7_LL y CB_d4_LL",25.805,2.398,26.827,27.523,23.065,3.824,3.670,0.736,0.035,True,1.281,3.248,entregas_ronda5/entrega_AICOLOMBIA_ENS_E10S_E7_CB_L...
1,5,ENS_E10S_E7_LL,Promedio fijo de E10S_LL y E7_LL,25.672,2.433,26.753,27.378,22.886,3.692,3.538,0.735,0.033,True,1.281,3.249,entregas_ronda5/entrega_AICOLOMBIA_ENS_E10S_E7_LL.csv
2,5,E10S_LL,E10S + lluvia (nivel y anomalías) en los 4 modelos,25.555,2.270,26.518,27.186,22.962,3.574,3.421,0.734,0.032,True,1.282,3.251,entregas_ronda5/entrega_AICOLOMBIA_E10S_LL.csv
3,5,CB_d4_LL,CatBoost prof. 4 con lluvia (clasificador); magnitu...,25.457,2.346,26.213,27.332,22.827,3.476,3.322,0.733,0.031,True,1.282,3.251,entregas_ronda5/entrega_AICOLOMBIA_CB_d4_LL.csv
4,5,E7_LL,E7_lento + lluvia (nivel y anomalías),25.369,2.603,26.532,27.188,22.387,3.388,3.234,0.732,0.031,True,1.282,3.262,entregas_ronda5/entrega_AICOLOMBIA_E7_LL.csv
5,5,E10S_LL_50,E10S_LL con 50 árboles en el clasificador,25.253,2.299,26.179,26.945,22.636,3.272,3.118,0.731,0.029,True,1.282,3.251,entregas_ronda5/entrega_AICOLOMBIA_E10S_LL_50.csv
6,5,E10S_LL_150,E10S_LL con 150 árboles en el clasificador,25.213,2.385,25.999,27.106,22.534,3.232,3.078,0.730,0.029,True,1.282,3.251,entregas_ronda5/entrega_AICOLOMBIA_E10S_LL_150.csv
7,5,E10S_LL_solo_clasif,Probabilidad de E10S_LL; magnitud y cuantiles de E10S,25.210,2.285,26.083,26.930,22.618,3.229,3.075,0.734,0.032,True,1.292,3.416,entregas_ronda5/entrega_AICOLOMBIA_E10S_LL_solo_cla...
8,5,E10S_LL_anom,E10S + solo anomalías de lluvia,22.991,2.359,23.213,25.231,20.529,1.010,0.856,0.711,0.010,True,1.290,3.408,entregas_ronda5/entrega_AICOLOMBIA_E10S_LL_anom.csv
9,5,E10S_LL_solo_magnitud,Probabilidad de E10S; magnitud y cuantiles de E10S_LL,22.326,2.456,22.151,24.865,19.963,0.345,0.191,0.701,0.000,False,1.282,3.251,entregas_ronda5/entrega_AICOLOMBIA_E10S_LL_solo_mag...


Detalle por corte (ronda 5):


auc               rmse_log               winkler                puntos                
corte                     F1     F2     F3       F1     F2     F3      F1     F2     F3      F1      F2      F3
variante                                                                                                       
CB_d4_LL               0.738  0.752  0.709    1.336  1.254  1.256   3.324  3.162  3.267  26.213  27.332  22.827
E10S                   0.697  0.727  0.680    1.352  1.260  1.263   3.501  3.295  3.450  21.716  24.608  19.618
E10S_LL                0.741  0.751  0.710    1.336  1.254  1.256   3.324  3.162  3.267  26.518  27.186  22.962
E10S_LL_150            0.736  0.750  0.706    1.336  1.254  1.256   3.324  3.162  3.267  25.999  27.106  22.534
E10S_LL_50             0.737  0.748  0.707    1.336  1.254  1.256   3.324  3.162  3.267  26.179  26.945  22.636
E10S_LL_anom           0.711  0.734  0.688    1.345  1.262  1.263   3.491  3.334  3.399  23.213  25.231  20.529
E10S_LL_placebo        0.699  0.724  0.679    1.348  1.259  1.263   3.520  3.320  3.422  21.886  24.284  19.570
E10S_LL_solo_clasif    0.741  0.751  0.710    1.352  1.260  1.263   3.501  3.295  3.450  26.083  26.930  22.618
E10S_LL_solo_magnitud  0.697  0.727  0.680    1.336  1.254  1.256   3.324  3.162  3.267  22.151  24.865  19.963
E10S_arboles_50        0.696  0.732  0.681    1.352  1.260  1.263   3.501  3.295  3.450  21.587  25.050  19.767
E7_LL                  0.742  0.750  0.704    1.338  1.255  1.254   3.351  3.128  3.308  26.532  27.188  22.387
E7_lento               0.701  0.723  0.672    1.351  1.260  1.268   3.499  3.262  3.475  22.057  24.199  18.697
ENS_E10S_E7            0.702  0.727  0.679    1.351  1.259  1.265   3.496  3.273  3.457  22.165  24.663  19.458
ENS_E10S_E7_CB_LL      0.744  0.753  0.711    1.336  1.253  1.254   3.327  3.142  3.275  26.827  27.523  23.065
ENS_E10S_E7_LL         0.743  0.752  0.709    1.336  1.253  1.254   3.331  3.135  3.282  26.753  27.378  22.886

Aporte de la lluvia (positivo = mejor; en RMSE y Winkler, negativo = mejor):


,variante,frente_a,que_mide,delta_puntos,delta_auc,delta_rmse_log,delta_winkler,delta_puntos_F1,delta_puntos_F2,delta_puntos_F3
0,E10S_LL,E10S,todas las variables de lluvia,3.574,0.032,-0.010,-0.164,4.802,2.577,3.344
1,E10S_LL_anom,E10S,solo anomalías,1.010,0.010,-0.002,-0.008,1.497,0.623,0.911
2,E10S_LL_placebo,E10S,PLACEBO (lluvia de hace 12 meses),-0.068,-0.001,-0.001,0.005,0.170,-0.324,-0.048
3,E10S_LL_50,E10S_arboles_50,50 árboles,3.118,0.028,-0.010,-0.164,4.592,1.894,2.869
4,E7_LL,E7_lento,E7 con lluvia,3.718,0.034,-0.011,-0.150,4.475,2.988,3.690
5,E10S_LL_solo_clasif,E10S,lluvia solo en la probabilidad,3.229,0.032,0.000,0.000,4.367,2.321,3.000
6,E10S_LL_solo_magnitud,E10S,lluvia solo en magnitud y cuantiles,0.345,0.000,-0.010,-0.164,0.435,0.256,0.345
7,ENS_E10S_E7_LL,ENS_E10S_E7,ensamble con lluvia,3.577,0.032,-0.010,-0.160,4.588,2.715,3.428


Anomalías reales +1.01 vs placebo -0.07 puntos → la ganancia sí viene de la lluvia del mes


c:\Users\ingju\AppData\Local\Programs\Python\Python312\Lib\site-packages\lightgbm\sklearn.py:1106: LGBMDeprecationWarning: The argument 'eval_set' is deprecated, use 'eval_X' and 'eval_y' instead.
  eval_set = _validate_eval_set_Xy(eval_set=eval_set, eval_X=eval_X, eval_y=eval_y)
c:\Users\ingju\AppData\Local\Programs\Python\Python312\Lib\site-packages\lightgbm\sklearn.py:1106: LGBMDeprecationWarning: The argument 'eval_set' is deprecated, use 'eval_X' and 'eval_y' instead.
  eval_set = _validate_eval_set_Xy(eval_set=eval_set, eval_X=eval_X, eval_y=eval_y)
c:\Users\ingju\AppData\Local\Programs\Python\Python312\Lib\site-packages\lightgbm\sklearn.py:1106: LGBMDeprecationWarning: The argument 'eval_set' is deprecated, use 'eval_X' and 'eval_y' instead.
  eval_set = _validate_eval_set_Xy(eval_set=eval_set, eval_X=eval_X, eval_y=eval_y)


Importancia de la lluvia en el clasificador de E10S_LL: 28.6% del total


,importancia
lluvia_log,0.236
log_poblacion,0.093
hist_tasa_mun,0.082
hist_tasa_mes,0.060
hist_reciente_12,0.049
oni_x_tasa,0.047
hist_esperado_log,0.045
hist_tasa_dep_mes,0.039
departamento,0.026
hist_tasa_dep,0.020


Mejora a E10S: ENS_E10S_E7_CB_LL (Promedio fijo de E10S_LL, E7_LL y CB_d4_LL) · 25.81 puntos (+3.82) · entregas_ronda5/entrega_AICOLOMBIA_ENS_E10S_E7_CB_LL.csv · ⚠ usa lluvia: entregar solo con confirmación de los organizadores


## 9. Registro central de experimentos

In [17]:
# Registro central de experimentos: una fila por variante en outputs/reportes/registro_experimentos.csv
import datetime
import hashlib
import json
import sys

NOTEBOOK = "notebook_experimentos5_lluvia_AICOLOMBIA"
REGISTRO_DRIVE = "/content/drive/MyDrive/Documentos Juan Felipe/Hackaton_PENUD/outputs/reportes"
COLUMNAS_REGISTRO = ["id_corrida", "fecha_hora", "ronda", "notebook", "entorno", "lightgbm", "huella_datos", "eventos_entrenamiento", "fuente", "variante", "descripcion", "config", "puntos_prom", "puntos_desv", "auc_prom", "rmse_log_prom", "winkler_prom", "cobertura_prom", "factor_q90", "archivo_entrega", "puntos_F1", "puntos_F2", "puntos_F3", "auc_F1", "auc_F2", "auc_F3", "rmse_log_F1", "rmse_log_F2", "rmse_log_F3", "winkler_F1", "winkler_F2", "winkler_F3", "cobertura_F1", "cobertura_F2", "cobertura_F3", "notas"]


def carpeta_registro():
    """Carpeta outputs/reportes del proyecto: en Drive (Colab) o local. None si no se encuentra."""
    if "google.colab" in sys.modules:
        try:
            from google.colab import drive
            if not os.path.isdir("/content/drive/MyDrive"):
                drive.mount("/content/drive")
        except Exception as error:
            print(f"⚠ No se pudo montar Drive: {error}")
        if os.path.isdir(REGISTRO_DRIVE):
            return REGISTRO_DRIVE
    for base in (os.getcwd(), os.path.dirname(os.getcwd())):
        if os.path.isdir(os.path.join(base, "outputs", "reportes")):
            return os.path.join(base, "outputs", "reportes")
    return None


ahora = datetime.datetime.now()
id_corrida = ahora.strftime("%Y%m%d_%H%M%S")
huella = hashlib.md5(f"{len(train)}|{int(train.tiene_evento.sum())}|{train.personas_desplazadas.sum():.0f}"
                     .encode()).hexdigest()[:8]
det = pd.DataFrame(detalle)
filas_registro = []
for r in resumen:
    d = det[det["variante"] == r["variante"]].set_index("corte")
    fila = {"id_corrida": id_corrida, "fecha_hora": f"{ahora:%Y-%m-%d %H:%M}", "ronda": r["ronda"], "notebook": NOTEBOOK,
            "entorno": "colab" if "google.colab" in sys.modules else "local", "lightgbm": lgb.__version__,
            "huella_datos": huella, "eventos_entrenamiento": int(train["tiene_evento"].sum()), "fuente": "ejecución",
            "variante": r["variante"], "descripcion": r["descripcion"],
            "config": json.dumps(CONFIGS.get(r["variante"], {}), default=str, sort_keys=True),
            "archivo_entrega": r["archivo"], "notas": NOTAS.get(r["variante"], ""),
            **{k: r.get(k) for k in ["puntos_prom", "puntos_desv", "auc_prom", "rmse_log_prom", "winkler_prom",
                                     "cobertura_prom", "factor_q90"]}}
    for c in ["F1", "F2", "F3"]:
        for m in ["puntos", "auc", "rmse_log", "winkler", "cobertura"]:
            fila[f"{m}_{c}"] = d.loc[c, m] if c in d.index else np.nan
    filas_registro.append(fila)
nuevo = pd.DataFrame(filas_registro).reindex(columns=COLUMNAS_REGISTRO)
nuevo.to_csv(f"{CARPETA}/registro_experimentos_{id_corrida}.csv", index=False)    # copia que viaja en el zip

carpeta = carpeta_registro()
if carpeta:
    ruta = os.path.join(carpeta, "registro_experimentos.csv")
    previo = pd.read_csv(ruta, dtype={"id_corrida": str}) if os.path.exists(ruta) else None
    total = nuevo if previo is None else pd.concat([previo, nuevo], ignore_index=True)
    total.to_csv(ruta, index=False)
    print(f"Registro central actualizado: +{len(nuevo)} filas → {ruta} ({len(total)} en total)")
else:
    print(f"No se encontró la carpeta del proyecto: el registro quedó en el zip ({CARPETA}/registro_experimentos_{id_corrida}.csv).")
    print("Para sumarlo al registro central: python scripts/consolidar_registro.py <ese archivo>")

Registro central actualizado: +15 filas → g:\My Drive\Documentos Juan Felipe\Hackaton_PENUD\outputs\reportes\registro_experimentos.csv (63 en total)


## 10. Guardar los CSV de la ronda 5

In [18]:
shutil.make_archive("entregas_ronda5", "zip", CARPETA)
try:
    from google.colab import files
    files.download("entregas_ronda5.zip")
except ImportError:
    print("Guardado en", os.path.abspath("entregas_ronda5.zip"))

Guardado en g:\My Drive\Documentos Juan Felipe\Hackaton_PENUD\notebooks\entregas_ronda5.zip
